In [ ]:
# Executing this cell will:

# Disable all TQDM outputs in stdout.
import os

os.environ["DISABLE_TQDM"] = "True"

# Setup the python logger for the Public API
from osekit import setup_logging

setup_logging()  # Overwrites the default logger to

# Creating the Public Project

[APLOSE](https://osmose.ifremer.fr/doc/)-compatible projects are build thanks to OSEkit's `Public API`.

First, we will build a project and run a transform that would be uploaded and annotated on APLOSE (see the [Public API documentation](https://project-osmose.github.io/OSEkit/publicapi_usage.html) for more info).

The `_static/annotations/aplose_results.csv` file used in this notebook simulates the results of this annotation campaign.

## Build the Project

First, we have to build the project from the raw audio files:

In [ ]:
from pathlib import Path
from osekit.public.project import Project
from osekit.core.instrument import Instrument

folder = Path(r"_static/sample_audio/timestamped")
strptime_format = r"%y%m%d_%H%M%S"

project = Project(
    folder=folder,
    strptime_format=strptime_format,
    instrument=Instrument(end_to_end_db=150.0),
    timezone="UTC",
)

project.build()

## Declare & Run the Transform

Then we **declare** and **run** a `Transform` which would export the spectrograms to be annotated:

In [ ]:
from osekit.public.transform import Transform, OutputType
from osekit.utils.audio import Normalization
from pandas import Timestamp, Timedelta
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import hamming

transform = Transform(
    output_type=OutputType.SPECTROGRAM,
    begin=Timestamp("2022-09-25 22:35:15+0000"),
    end=Timestamp("2022-09-25 22:36:25+0000"),
    data_duration=Timedelta(seconds=7.5),
    normalization=Normalization.DC_REJECT,
    fft=ShortTimeFFT(win=hamming(1024), hop=128, fs=project.origin_dataset.sample_rate),
    v_lim=(50.0, 120.0),  # Boundaries of the spectrograms
    colormap="viridis",  # Default value
    name="example_transform",
)

# We remove all spectrograms that contain silent parts
ads = project.prepare_audio(transform=transform)
ads.remove_empty_data(threshold=0.99)

project.run(transform=transform, audio_dataset=ads)

# Using APLOSE annotation results [^download]

[^download]: This notebook can be downloaded as **{nb-download}`example_aplose_result.ipynb`**.

Parse the annotation result csv file thanks to the `Annotation` class:

In [ ]:
from pathlib import Path
from osekit.core.annotation import Annotation

annotations = Annotation.from_csv(csv=Path(r"_static/annotations/aplose_results.csv"))

## Filtering the annotations

We can use basic python filtering to access specific annotations.

Here, we will:
- keep only `Odontocete whistle` annotations of type `BOX` with a strong **confidence level**
- Filter the `SpectroDataset` to keep only the files in which such annotations were made
- Plot the annotations as `Rectangle`s on the spectrograms.

### Keeping specific annotations

Filtering out unwanted annotations can be done with a basic list comprehension:

In [ ]:
def does_satisfy_constraints(annotation: Annotation) -> bool:
    # Keeping only odontocete whistles
    if annotation.label != "Odontocete whistle":
        return False

    # Keeping only BOX annotations
    if annotation.type != "BOX":
        return False

    # Keeping only maximum confidence level annotations
    if (
        annotation.confidence_indicator.level
        < annotation.confidence_indicator.maximum_level
    ):
        return False

    return True


filtered_annotations = [
    annotation for annotation in annotations if does_satisfy_constraints(annotation)
]

## Filtering the SpectroDataset

`Annotation`s inherit from the `Event` class, which allows for an easy filtering of the `SpectroData`:

In [ ]:
# Recover the transform output (SpectroDataset)
sds = project.get_output(output_name="example_transform")

# Keeping only SpectroDatas that contain filtered annotations
sds.data = [
    sd
    for sd in sds.data
    if any(annotation.overlaps(sd) for annotation in filtered_annotations)
]

## Plotting the Spectrograms along with annotations

We then want to plot annotation boxes directly the spectrograms:

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with one spectrogram per row
fig, axs = plt.subplots(nrows=len(sds.data), ncols=1)

# Plot spectrograms
for idx, sd in enumerate(sds.data):
    # We want to plot each spectrogram in a specific ax
    ax = axs[idx]

    sd.plot(ax=ax)

    # We want to plot all annotations related to this spectrogram
    for annotation in filtered_annotations:
        if not annotation.overlaps(sd):
            continue

        # Annotations are plotted as matplotlib Rectangles
        rectangle = annotation.to_rectangle(fill=False)
        ax.add_patch(rectangle)

# Let's take a look at the output figure
plt.show()

In [ ]:
# Reset the project to get all files back to place.
project.reset()